# Domovina.tv — Canary transkripcija + pyannote diarizacija (BATCH) *(Colab)*

Batch obrada svih `.wav` datoteka koje su uploadane na Google Drive (`canary_wav` folder) — generira `.canary.srt`, `.canary.csv` i `.canary.diarized.srt` za svaki podcast.

## Što ovaj notebook radi

```
Google Drive (canary_wav/)              Google Colab (T4/L4/G4 GPU)
┌────────────────────────────┐          ┌─────────────────────────────────┐
│ kanal_x/                   │  mount   │ FAZA 1: transcribe_canary.py    │
│   ep001.wav        ───────────────►   │   nvidia/canary-1b-v2 (BF16)    │
│                            │          │   → .canary.srt + .canary.csv   │
│   ep001.wav.canary.srt ◄───────────── │                                 │
│                            │          │ FAZA 2: diarize_canary.py       │
│                            │          │   pyannote community-1          │
│                            │          │   exclusive_speaker_diarization │
│   ep001.wav.canary.        │          │   + distributed lock            │
│     diarized.srt   ◄───────────────── │                                 │
└────────────────────────────┘          └─────────────────────────────────┘
```

Obje faze su **idempotentne** — preskaču datoteke koje već imaju output. Možeš pokretati notebook neograničeno; obrađuju se samo nove epizode.

## Kako koristiti (jednom)

1. **GPU runtime**: Runtime → Change runtime type → odaberi `T4` (free), `L4` (Pro) ili `A100` (Pro+).
2. **HF token**: lijevi panel → ključić (Secrets) → dodaj `HF_TOKEN` (uključi *Notebook access*).
   - Token: https://huggingface.co/settings/tokens
   - Prihvati uvjete: https://huggingface.co/pyannote/speaker-diarization-community-1
3. **Drive folder**: WAV-ovi moraju već biti uploadani u `MyDrive/domovina_fetch_data/canary_wav/{kanal}/...wav` (lokalni `run_pipeline.sh` to radi automatski u koraku 2.5).
4. **Runtime → Run all** (⌘/Ctrl+F9). Prvi put će se kernel jednom restartati nakon instalacije — to je očekivano. Klikni *Run all* još jednom.

## Resursi

| GPU | Canary 1B v2 | pyannote community-1 | Per file (75 min audio) |
|---|---|---|---|
| T4 (16 GB) | ~30-50s | ~30-60s VRAM peak ~9.5 GB | ~1.5 min |
| L4 / G4 (24 GB) | ~10-20s | ~20-40s | ~45-60s |
| A100 (40 GB) | ~5-10s | ~15-30s | ~25-40s |

VRAM headroom je dovoljan na T4 jer notebook **prvo skida Canary** prije učitavanja pyannote.

## Multi-machine paralelizam

Ako pokrećeš isti notebook iz više Colaba istovremeno (npr. više besplatnih sesija), svaki worker stvori `.canary.lock` na Drive-u prije obrade fajla — drugi ga preskoče. Uključuje se setanjem `USE_DISTRIBUTED_LOCK = True` u konfiguraciji.

---


## 0. Konfiguracija

Sve postavke koje ćeš mijenjati su ovdje. Niže ćelije čitaju ove varijable.


In [ ]:
# ─── Drive locations ─────────────────────────────────────────────────────────
DRIVE_MOUNT_POINT = "/content/drive"
DRIVE_DATA_DIR    = "MyDrive/domovina_fetch_data/canary_wav"   # gdje su WAV-ovi
INPUT_DIR         = f"{DRIVE_MOUNT_POINT}/{DRIVE_DATA_DIR}"

# ─── Što pokrenuti ───────────────────────────────────────────────────────────
RUN_TRANSCRIPTION = True   # Faza 1: WAV → .canary.srt + .canary.csv
RUN_DIARIZATION   = True   # Faza 2: WAV + .canary.srt → .canary.diarized.srt

# ─── Batch limits ────────────────────────────────────────────────────────────
LIMIT             = None   # int ili None — npr. 5 za testiranje, None za sve
DRY_RUN           = False  # True: samo prikaz, bez obrade

# ─── Pyannote speaker hints (opcionalno) ─────────────────────────────────────
# Ako znaš tipičan broj govornika za većinu podcasta, postavi raspon.
# None = auto-detekcija (sigurno za heterogen korpus s različitim formatima).
MIN_SPEAKERS      = None   # npr. 2
MAX_SPEAKERS      = None   # npr. 6

# ─── Multi-machine koordinacija ──────────────────────────────────────────────
# True ako pokrećeš notebook iz više Colab sesija paralelno.
# Stvori .canary.lock na Drive-u; drugi workeri preskoče zaključane fajlove.
USE_DISTRIBUTED_LOCK = False

# ─── Repo ────────────────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/domovinatv/fetch.domovina.tv.git"
REPO_PATH = "/content/fetch.domovina.tv"

print("Konfiguracija učitana.")
print(f"  Input:           {INPUT_DIR}")
print(f"  Transkripcija:   {RUN_TRANSCRIPTION}")
print(f"  Diarizacija:     {RUN_DIARIZATION}")
print(f"  Limit:           {LIMIT if LIMIT else 'sve'}")
print(f"  Dry run:         {DRY_RUN}")
print(f"  Distributed lock:{USE_DISTRIBUTED_LOCK}")


## 1. GPU provjera

Ako padne, idi na *Runtime → Change runtime type → T4 GPU* i pokreni *Run all* ponovo.


In [ ]:
import subprocess
try:
    out = subprocess.check_output(["nvidia-smi", "-L"]).decode().strip()
    print(out)
    # Detalji o memoriji
    mem = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.total,memory.free", "--format=csv,noheader,nounits"]
    ).decode().strip()
    total, free = [int(x.strip()) for x in mem.split(",")]
    print(f"\nVRAM: {free} MB slobodno / {total} MB ukupno ({free/total*100:.0f}%)")
    if total < 14000:
        print("\nUPOZORENJE: Manje od 14 GB VRAM — pyannote community-1 može OOM. Razmisli o L4/A100.")
except Exception as e:
    raise RuntimeError(
        "GPU nije dostupan. Idi na Runtime → Change runtime type → T4/L4/A100 GPU."
    ) from e


## 2. Mount Google Drive

Tražit će autorizaciju u browseru — odobri.


In [ ]:
from google.colab import drive
drive.mount(DRIVE_MOUNT_POINT)

import os
assert os.path.isdir(INPUT_DIR), (
    f"Direktorij ne postoji: {INPUT_DIR}\n"
    f"Provjeri DRIVE_DATA_DIR u konfiguraciji ili da li su WAV-ovi uploadani."
)
print(f"Drive mountan. INPUT_DIR sadrži: {len(os.listdir(INPUT_DIR))} entry-ja.")


## 3. Clone / pull repo

Notebook se oslanja na `colab_canary/transcribe_canary.py` i `colab_diarize/diarize_canary.py` iz repoa — oni su workhorses za batch obradu (lockovi, ETA, idempotentnost).


In [ ]:
import os, subprocess

if not os.path.isdir(REPO_PATH):
    print(f"Kloniram repo u {REPO_PATH}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    print(f"Repo postoji, povlačim najnovije...")
    subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], check=True)

# Verificiraj da postoje skripte
TRANSCRIBE_SCRIPT = f"{REPO_PATH}/colab_canary/transcribe_canary.py"
DIARIZE_SCRIPT    = f"{REPO_PATH}/colab_diarize/diarize_canary.py"
for s in (TRANSCRIBE_SCRIPT, DIARIZE_SCRIPT):
    assert os.path.isfile(s), f"Nedostaje skripta: {s}"
print(f"Skripte spremne: {os.path.basename(TRANSCRIBE_SCRIPT)}, {os.path.basename(DIARIZE_SCRIPT)}")


## 4. Instalacija dependencija (~2-3 min)

Instalira NeMo (Canary), pyannote.audio (diarizacija), gdown (download helper). Prvi put ova ćelija prisilno gasi kernel kako bi novi numpy/scipy proradili (inače `ImportError: _center` u NeMo). Nakon restarta klikni **Runtime → Run all** ponovno — drugi put preskače instalaciju i sve teče do kraja.


In [ ]:
import os

MARKER = "/content/.domovina_deps_ok"

def _have_deps():
    try:
        import nemo.collections.asr  # noqa: F401
        from pyannote.audio import Pipeline  # noqa: F401
        import soundfile  # noqa: F401
        return True
    except Exception as e:
        print(f"deps check: {type(e).__name__}: {e}")
        return False

if os.path.exists(MARKER) and _have_deps():
    print("Dependencies već instalirani i spremni.")
else:
    print("Instaliram dependencies — traje ~2-3 min...")
    get_ipython().system(
        'pip install -qU numpy "nemo_toolkit[asr]" "pyannote.audio>=4.0.0" '
        'soundfile 2>&1 | tail -5'
    )
    open(MARKER, "w").write("ok")
    print("")
    print("=" * 60)
    print(" Instalacija gotova — gasim kernel radi čistog reloada.")
    print(" → Nakon restarta klikni Runtime → Run all ponovno.")
    print("=" * 60)
    import time
    time.sleep(2)
    os.kill(os.getpid(), 9)


In [ ]:
# Bezopasni shimovi za starije import path-eve (NeMo + huggingface_hub)
import sys
class _DummyYTTM: pass
sys.modules.setdefault("youtokentome", _DummyYTTM)

import huggingface_hub
if getattr(huggingface_hub, "ModelFilter", None) is None:
    class ModelFilter: pass
    huggingface_hub.ModelFilter = ModelFilter

import torch
print("torch:", torch.__version__,
      "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


## 5. HuggingFace token

Token se učita iz Colab Secrets (`HF_TOKEN`). Ako secret nije postavljen, pita za upis u runtime (neće se sačuvati).


In [ ]:
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN učitan iz Colab Secrets.")
except Exception:
    pass

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Upiši HuggingFace token (hf_...): ").strip()

assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "HF_TOKEN nije valjan (mora počinjati s 'hf_')"

# Postavi za subprocess pozive (transcribe_canary.py + diarize_canary.py ga čitaju iz env)
os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN spreman za korištenje.")


## 6. Pregled posla (dry run)

Skenira Drive i prikazuje:
- Koliko WAV datoteka postoji ukupno
- Koliko već ima `.canary.srt` (transkripcija gotova)
- Koliko već ima `.canary.diarized.srt` (diarizacija gotova)
- Koliko će se obraditi u ovom runu

Ova ćelija ne radi nikakvu obradu — samo prikazuje.


In [ ]:
def scan_progress(input_dir):
    wavs, with_srt, with_diar = [], [], []
    for root, _, files in os.walk(input_dir, followlinks=True):
        for f in files:
            if f.startswith("._") or not f.endswith(".wav"):
                continue
            p = os.path.join(root, f)
            wavs.append(p)
            if os.path.exists(p + ".canary.srt"):
                with_srt.append(p)
            if os.path.exists(p + ".canary.diarized.srt"):
                with_diar.append(p)
    return wavs, with_srt, with_diar

print("Skeniram Drive (može potrajati ~30s za veliki korpus)...")
wavs, with_srt, with_diar = scan_progress(INPUT_DIR)

to_transcribe = [w for w in wavs if not os.path.exists(w + ".canary.srt")]
to_diarize    = [w for w in with_srt if not os.path.exists(w + ".canary.diarized.srt")]

print(f"\n  Ukupno WAV datoteka:        {len(wavs)}")
print(f"  Već transkribirano (.srt):  {len(with_srt)} ({len(with_srt)/max(len(wavs),1)*100:.1f}%)")
print(f"  Već diarized (.diarized):   {len(with_diar)} ({len(with_diar)/max(len(wavs),1)*100:.1f}%)")
print(f"\n  Za transkripciju u ovom runu: {len(to_transcribe)}")
print(f"  Za diarizaciju u ovom runu:   {len(to_diarize)}")

if LIMIT:
    print(f"\n  LIMIT={LIMIT} → obradit će se najviše {LIMIT} datoteka po fazi")

# Procjena trajanja (tipično 75 min audio na T4: ~1.5 min/file za obje faze)
gpu_name = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else ""
if "a100" in gpu_name:
    sec_per_file = 30
elif "l4" in gpu_name or "g4" in gpu_name:
    sec_per_file = 50
else:
    sec_per_file = 90  # T4 / V100
n_to_run = min(LIMIT or 10**9, len(to_transcribe) if RUN_TRANSCRIPTION else 0) + \
           min(LIMIT or 10**9, len(to_diarize) if RUN_DIARIZATION else 0)
if n_to_run:
    eta_min = n_to_run * sec_per_file / 60
    print(f"\n  Procjena trajanja na ovom GPU-u: ~{eta_min:.0f} min")
    print(f"  (heuristika: ~{sec_per_file}s po fileu × {n_to_run} obrada)")


## 7. FAZA 1 — Canary 1B v2 transkripcija

Pokreće `transcribe_canary.py` u batch modu. Skripta:
- Učita Canary 1B v2 model (BF16 ako GPU podržava — pola memorije, brži)
- Skenira `INPUT_DIR` rekurzivno za `.wav` datoteke
- Preskače sve koje već imaju `.canary.srt`
- Za svaku generira `.canary.srt` i `.canary.csv`
- Heartbeat svakih 60s tijekom dugačkih datoteka

Output ide u **isti folder** kao WAV (na Drive-u), tako da se odmah vidi i sinkronizira nazad lokalno preko `rclone`.


In [ ]:
import shlex

if not RUN_TRANSCRIPTION:
    print("Preskačem FAZU 1 (RUN_TRANSCRIPTION=False).")
else:
    cmd = [
        "python", "-u", TRANSCRIBE_SCRIPT,
        "--input-dir", INPUT_DIR,
    ]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    if DRY_RUN:
        cmd += ["--dry-run"]

    print(">", " ".join(shlex.quote(c) for c in cmd))
    print()
    # Stream output u realnom vremenu
    get_ipython().system(" ".join(shlex.quote(c) for c in cmd))


## 8. Oslobodi VRAM između faza

Canary model više nije potreban — pyannote treba ~9.5 GB VRAM peak. Restart kernela bio bi sigurniji ali bi izgubio sve varijable; umjesto toga čistimo cache i puštamo OS da reclaim-a memoriju kad sljedeći subprocess (`diarize_canary.py`) startuje.


In [ ]:
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"VRAM slobodno: {free_gb:.2f} GB")


## 9. FAZA 2 — pyannote community-1 diarizacija

Pokreće `diarize_canary.py`. Skripta:
- Učita pyannote/speaker-diarization-community-1 (~9.5 GB VRAM)
- Skenira `INPUT_DIR` za WAV-ove koji imaju `.canary.srt` ali nemaju `.canary.diarized.srt`
- Koristi **exclusive_speaker_diarization** mode → jedan govornik u svakom trenutku (bez overlapa, idealno za SRT alignment)
- Spaja govornike s SRT segmentima preko najveće vremenske preklapanosti
- Ako je `USE_DISTRIBUTED_LOCK=True`, koordinira s drugim Colab sesijama preko `.canary.lock` fajlova

Output: `{wav_basename}.canary.diarized.srt` u istom folderu.


In [ ]:
import shlex

if not RUN_DIARIZATION:
    print("Preskačem FAZU 2 (RUN_DIARIZATION=False).")
else:
    cmd = [
        "python", "-u", DIARIZE_SCRIPT,
        "--input-dir", INPUT_DIR,
    ]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]
    if DRY_RUN:
        cmd += ["--dry-run"]
    if MIN_SPEAKERS is not None:
        cmd += ["--min-speakers", str(MIN_SPEAKERS)]
    if MAX_SPEAKERS is not None:
        cmd += ["--max-speakers", str(MAX_SPEAKERS)]
    if USE_DISTRIBUTED_LOCK:
        # --drive-mount: lock fajlovi se stvaraju direktno na mountanom Drive-u
        cmd += ["--drive-mount", INPUT_DIR]

    print(">", " ".join(shlex.quote(c) for c in cmd))
    print()
    get_ipython().system(" ".join(shlex.quote(c) for c in cmd))


## 10. Sažetak — što je novo na Drive-u

Ponovno skenira Drive i pokazuje delta — koliko `.canary.srt` i `.canary.diarized.srt` je generirano u ovom runu. Ovaj broj odgovara onome što ćeš dobiti lokalno kad sljedeći put pokreneš `run_pipeline.sh` (korak 0 ih `rclone`-om povuče).


In [ ]:
wavs2, with_srt2, with_diar2 = scan_progress(INPUT_DIR)

new_srt  = len(with_srt2)  - len(with_srt)
new_diar = len(with_diar2) - len(with_diar)

print("─" * 60)
print(f"  Novih .canary.srt:           +{new_srt}")
print(f"  Novih .canary.diarized.srt:  +{new_diar}")
print("─" * 60)
print(f"  Stanje na Drive-u:")
print(f"    {len(wavs2)} WAV ukupno")
print(f"    {len(with_srt2)}/{len(wavs2)} transkribirano ({len(with_srt2)/max(len(wavs2),1)*100:.1f}%)")
print(f"    {len(with_diar2)}/{len(wavs2)} diarized   ({len(with_diar2)/max(len(wavs2),1)*100:.1f}%)")
print()
print("Sljedeći korak na lokalnom stroju:")
print("  ./run_pipeline.sh   # korak 0 rclone-om povuče nove .canary.diarized.srt")
